In [2]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
from typing import Union

from google.cloud import secretmanager

# set the Google Cloud project if env is set, otherwise default to rxrx-medchem-auto-dev
google_cloud_project = os.getenv("GOOGLE_CLOUD_PROJECT", "rxrx-medchem-auto-dev")


def access_secret_version(
    secret_name: str,
    project_id: str = google_cloud_project,
    secret_ver: Union[int, str] = "latest",  # noqa
) -> str:
    """
    Accesses a secret version from Google Cloud Secret Manager.

    Args:
        secret_name (str): The name of the secret.
        project_id (str, optional): The project ID. Defaults to GOOGLE_CLOUD_PROJECT env var or "rxrx-medchem-auto-dev".

        secret_ver (Union[int, str], optional): The version of the secret to access. Defaults to "latest".

    Returns:
        str: The secret data.

    Examples:
        # Access the latest version of the secret
        >>> access_secret_version("my_secret")
        'my_secret_data'

        # Access a specific version of the secret
        >>> access_secret_version("my_secret", secret_ver=2)
        'my_secret_data_v2'
    """
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_name}/versions/{secret_ver}"
    response = client.access_secret_version(request={"name": name})
    data = response.payload.data.decode("UTF-8")
    assert data is not None, "Secret not found"  # noqa
    return data


def set_env_secrets(secret_list: list[str]):
    for secret in secret_list:
        secret_data = access_secret_version(secret)
        os.environ[secret] = secret_data


def set_env_secrets_safely(secret_list: list[str]):
    from loguru import logger

    try:
        set_env_secrets(["OPENAI_API_KEY", "LANGCHAIN_API_KEY"])
    except Exception as e:
        import os
        import sys

        logger.error(f"Error setting environment secrets {e}")
        is_in_ci = os.getenv("CF_REPO_NAME", None)
        if not is_in_ci:
            sys.exit(1)

In [4]:
set_env_secrets_safely(["OPENAI_API_KEY", "LANGCHAIN_API_KEY"])

In [19]:
! cat chain.pem

In [15]:
from explain.literature._esearch_utils import search_indexes

keywords = ["EGFR", "Imatinib"]
es_keywords = list(set(keywords))

articles = await search_indexes(
    keywords=es_keywords, indexes="full", top_k=10, primitive_catalog=None
)



{'size': 10, '_source': True, 'query': {'bool': {'must': [{'nested': {'path': 'sections', 'query': {'bool': {'must': [{'bool': {'should': [{'match_phrase': {'sections.text': {'query': 'Imatinib', 'boost': 5}}}, {'match_phrase': {'sections.text': {'query': 'EGFR', 'boost': 5}}}], 'minimum_should_match': 1}}]}}, 'inner_hits': {'highlight': {'fields': {'sections.text': {'fragment_size': 1000, 'number_of_fragments': 4, 'require_field_match': True}}, 'pre_tags': ['<em>'], 'post_tags': ['</em>']}}}}], 'must_not': [{'exists': {'field': 'retraction_reasons'}}]}}}
{'size': 10, '_source': True, 'query': {'bool': {'must': [{'nested': {'path': 'sections', 'query': {'bool': {'must': [{'bool': {'should': [{'match_phrase': {'sections.text': {'query': 'Imatinib', 'boost': 5}}}, {'match_phrase': {'sections.text': {'query': 'EGFR', 'boost': 5}}}], 'minimum_should_match': 1}}]}}, 'inner_hits': {'highlight': {'fields': {'sections.text': {'fragment_size': 1000, 'number_of_fragments': 4, 'require_field_matc

2025-08-25 10:33:11.224 | ERROR    | explain.literature._esearch_utils:retrieve_article:294 - Error performing search: TLS error caused by: TlsError(TLS error caused by: ClientConnectorCertificateError(Cannot connect to host elasticsearch.centaur-platform-dev.com:443 ssl:True [SSLCertVerificationError: (1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)')]))
2025-08-25 10:33:11.224 | ERROR    | explain.literature._esearch_utils:retrieve_article:295 - Full exception details:
Traceback (most recent call last):

  File "/Users/emmanuel.noutahi/miniconda3/envs/dev/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x103419ce0, file "/Users/emmanuel.noutahi/Code/h

In [ ]:
body = await create_search_body()

In [10]:
results = await retrieve_articles(body)

2025-08-25 10:28:33.807 | ERROR    | __main__:retrieve_articles:79 - Error performing search: TLS error caused by: TlsError(TLS error caused by: ClientConnectorCertificateError(Cannot connect to host elasticsearch.centaur-platform-dev.com:443 ssl:True [SSLCertVerificationError: (1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)')]))
2025-08-25 10:28:33.808 | ERROR    | __main__:retrieve_articles:80 - Full exception details:
Traceback (most recent call last):

  File "/Users/emmanuel.noutahi/miniconda3/envs/dev/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x103419ce0, file "/Users/emmanuel.noutahi/Code/hooke-explain/.venv/lib/python3.12/site-packages/ip